# Distributed Training on Kaggle (2x T4 GPUs)

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Clone Repository and Install

In [ ]:
!git clone -b distributed_LM https://github.com/0Chris5R/StanfordCS336-Own-Transformer.git assignment1-basics

In [ ]:
!pip install -e assignment1-basics

In [ ]:
import torch
import cs336_basics
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 3. Download Data

In [ ]:
!mkdir -p data
!cp /kaggle/input/your-dataset/owt_train.npy data/
!cp /kaggle/input/your-dataset/owt_valid.npy data/
!ls -la data/

## 4. Run Distributed Training

In [ ]:
%cd assignment1-basics

In [ ]:
%%writefile run_training.py
import torch
from cs336_systems.train_distributed import run_distributed_training

run_distributed_training(
    world_size=2,
    d_model=1280,
    num_layers=24,
    num_heads=10,
    d_ff=3456,
    vocab_size=32000,
    context_length=1024,
    rope_theta=10000,
    steps=5000,
    batch_size=32,
    max_learning_rate=5e-4,
    weight_decay=0.01,
    betas=(0.9, 0.95),
    shard_optimizer=True,
    shard_gradient=True,
    use_muon=True,
    cautious_weight_decay=True,
    mixed_precision_dtype=torch.float16,
    train_path="../data/owt_train.npy",
    val_path="../data/owt_valid.npy",
    tokenizer_path="checkpoints/tokenizer_owt.model",
    save_model_path="checkpoints/distributed_model",
    val_interval=250,
    save_interval=1000,
)

In [ ]:
!python run_training.py

## 5. Save Checkpoints

In [ ]:
!cp -r checkpoints/*.pt /kaggle/working/ 2>/dev/null || echo "No checkpoints saved yet"
!ls -la /kaggle/working/